# YOLOv11 Local Training Notebook (v2)
## Real-time Student Behavior Detection (Local)

This notebook follows the same flow as train_model.ipynb, with local-first settings for Windows/venv runs.

## 1. Import Required Libraries and Setup

In [ ]:
import sys, platform
print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())

In [ ]:
%pip install -q ultralytics opencv-python pyyaml matplotlib pillow

In [ ]:
from pathlib import Path
import json
import yaml
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

PROJECT_ROOT = Path('.').resolve()

dataset_candidates = [
    PROJECT_ROOT / 'dataset',
    Path('dataset').resolve(),
]

resolved_dataset = None
for candidate in dataset_candidates:
    if (candidate / 'data.yaml').exists():
        resolved_dataset = candidate
        break

if resolved_dataset is None:
    resolved_dataset = PROJECT_ROOT / 'dataset'

DATASET_ROOT = resolved_dataset
DATA_YAML = DATASET_ROOT / 'data.yaml'

RUNS_ROOT = PROJECT_ROOT / 'fyp_runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUNS_ROOT / 'classroom_model_v2'
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'

device = '0' if torch.cuda.is_available() else 'cpu'
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset root: {DATASET_ROOT}')
print(f'Data yaml: {DATA_YAML}')
print(f'Runs root: {RUNS_ROOT}')
print(f'PyTorch version: {torch.__version__}')
print(f'torch.cuda.is_available(): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device name: {torch.cuda.get_device_name(0)}')
else:
    print('CUDA GPU not detected. Training will run on CPU.')
print(f'Training device setting: {device}')

if not DATA_YAML.exists():
    print('Dataset not found. Expected dataset/data.yaml in project root.')

## 2. Validate Dataset Structure

In [ ]:
expected_dirs = [
    DATASET_ROOT / 'train' / 'images',
    DATASET_ROOT / 'train' / 'labels',
    DATASET_ROOT / 'valid' / 'images',
    DATASET_ROOT / 'valid' / 'labels',
    DATASET_ROOT / 'test' / 'images',
    DATASET_ROOT / 'test' / 'labels',
]

print('Dataset directory check:')
all_ok = True
for p in expected_dirs:
    ok = p.exists()
    print(f"{'OK ' if ok else 'MISS'} - {p}")
    all_ok = all_ok and ok

if not all_ok:
    raise FileNotFoundError('One or more required dataset folders are missing. Fix dataset structure first.')

print('All required dataset directories exist.')

## 3. Configure Dataset YAML

In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(f'Missing file: {DATA_YAML}')

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

if not isinstance(data_cfg, dict):
    raise ValueError('data.yaml is not a valid mapping/dictionary.')

expected_paths = {
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
}

changed = False
resolved_root = str(DATASET_ROOT.resolve())
if data_cfg.get('path') != resolved_root:
    data_cfg['path'] = resolved_root
    changed = True

for k, v in expected_paths.items():
    if data_cfg.get(k) != v:
        data_cfg[k] = v
        changed = True

if changed:
    with open(DATA_YAML, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data_cfg, f, sort_keys=False)
    print('Updated data.yaml path/split entries for current runtime.')

print('Loaded data.yaml:')
print(json.dumps({
    'path': data_cfg.get('path'),
    'train': data_cfg.get('train'),
    'val': data_cfg.get('val'),
    'test': data_cfg.get('test'),
    'nc': data_cfg.get('nc'),
    'names': data_cfg.get('names'),
}, indent=2))

expected_names = ['handrise', 'read', 'write', 'sleep', 'using_device', 'stand', 'look_forward', 'turn_head']

names = data_cfg.get('names', [])
nc = int(data_cfg.get('nc', -1))
if nc != len(names):
    raise ValueError(f"nc ({nc}) does not match len(names) ({len(names)})")

if list(names) != expected_names:
    print('Warning: class names differ from expected master labels.')
    print('Expected:', expected_names)
    print('Found   :', names)
else:
    print('Class labels match expected master labels.')

print('data.yaml validation complete.')

## 4. Train YOLOv11 Model

In [ ]:
DO_TRAIN = False

if not DO_TRAIN:
    print('Training skipped. Set DO_TRAIN=True to start training locally.')
else:
    model = YOLO('yolo11m.pt')

    QUICK_RUN = True

    if QUICK_RUN:
        epochs = 10
        imgsz = 512
        batch = 8
        patience = 8
    else:
        epochs = 100
        imgsz = 640
        batch = -1 if device != 'cpu' else 8
        patience = 20

    train_config = {
        'data': str(DATA_YAML),
        'epochs': epochs,
        'imgsz': imgsz,
        'batch': batch,
        'device': device,
        'project': str(RUNS_ROOT),
        'name': 'classroom_model_v2',
        'verbose': True,
        'save': True,
        'workers': 2,
        'cache': False,
        'amp': True,
        'optimizer': 'auto',
        'cos_lr': True,
        'patience': patience,
        'plots': True,
        'close_mosaic': 10,
        'seed': 42,
    }

    print('Training config:')
    for k, v in train_config.items():
        print(f'  {k}: {v}')

    try:
        train_results = model.train(**train_config)
        print('Training complete.')
    except RuntimeError as e:
        msg = str(e).lower()
        if 'out of memory' in msg or 'cudnn' in msg:
            print('OOM/accelerator error detected. Rerun with batch=8 if needed.')
        raise

    print(f'Expected best weights at: {BEST_WEIGHTS}')

## 5. Test the Trained Model

This cell runs evaluation on the held-out test split using the best weights saved during training.

In [ ]:
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f'Missing trained weights: {BEST_WEIGHTS}. Run the training cell first.')

test_model = YOLO(str(BEST_WEIGHTS))

test_config = {
    'data': str(DATA_YAML),
    'split': 'test',
    'imgsz': 640,
    'batch': 16,
    'device': device,
    'project': str(RUNS_ROOT),
    'name': 'classroom_model_v2_test',
    'verbose': True,
    'save': True,
    'plots': True,
}

print('Testing config:')
for k, v in test_config.items():
    print(f'  {k}: {v}')

test_results = test_model.val(**test_config)
print('Testing complete.')

results_dir = Path(getattr(test_results, 'save_dir', RUNS_ROOT / 'classroom_model_v2_test'))
print(f'Test artifacts saved to: {results_dir}')

if hasattr(test_results, 'results_dict'):
    print('Metrics:')
    print(json.dumps(test_results.results_dict, indent=2, default=str))

confusion_matrix_path = results_dir / 'confusion_matrix.png'
results_png_path = results_dir / 'results.png'
print(f'Confusion matrix exists: {confusion_matrix_path.exists()}')
print(f'Results plot exists: {results_png_path.exists()}')

## 6. Visualize Training Results

In [ ]:
results_png = RUN_DIR / 'results.png'
cm_png = RUN_DIR / 'confusion_matrix.png'
val_batch_png = RUN_DIR / 'val_batch0_pred.jpg'

print('Artifacts:')
for p in [results_png, cm_png, val_batch_png]:
    print(f"{'OK ' if p.exists() else 'MISS'} - {p}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
images = [results_png, cm_png, val_batch_png]
titles = ['Training Curves', 'Confusion Matrix', 'Sample Validation Predictions']

for ax, img_path, title in zip(axes, images, titles):
    ax.set_title(title)
    ax.axis('off')
    if img_path.exists():
        img = Image.open(img_path)
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, 'Not generated yet', ha='center', va='center')

plt.tight_layout()
plt.show()